# 5b단계: YOLO 파인튜닝 + 평가

5a에서 만들고 Roboflow에서 고친 학습 정답으로 **YOLO26을 직접 파인튜닝**하고, 시험지 20장으로 기존 결과와 비교합니다.

**준비물**: Drive의 `pipeline_outputs/eval/finetune/` 폴더에 `images/`(5a가 만든 것)와 `finetune_gt.zip`(Roboflow 내보내기)

**실행 순서:** 위에서부터 끝까지 · **런타임: T4 GPU** · 전체 약 40~60분

## 공통 설정

In [ ]:
!pip install -q ultralytics
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

# 한글 글꼴 (그래프의 한글이 네모로 깨지지 않게)
import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

## ByteTrack
v3는 Ultralytics 안의 ByteTrack을 썼어요. 여기서는 저장해둔 탐지 결과에 설정만 바꿔 여러 번 돌리려고,
같은 규칙을 numpy로 옮긴 버전을 씁니다. 셀 끝의 자체 점검이 통과하는지 확인하세요.

In [ ]:
# ===== ByteTrack (논문 · Ultralytics 구현과 같은 규칙을 그대로 옮긴 버전) =====
# 저장해둔 탐지 결과에 바로 돌릴 수 있도록 numpy만으로 작성했습니다.
#   1) 칼만 필터로 기존 트랙의 이번 위치 예측
#   2) 높은 점수 박스와 IoU 짝짓기 (헝가리안)
#   3) 짝 못 찾은 트랙은 낮은 점수 박스와 한 번 더 짝짓기  ← ByteTrack의 핵심
#   4) 남은 높은 점수 박스 중 new_track_thresh 이상만 새 트랙으로 등록
#   5) track_buffer 프레임 동안 못 찾은 트랙은 삭제

class KalmanXYAH:
    """상태: [cx, cy, 종횡비 a, 높이 h, 각 속도 4개]. 등속 모델."""
    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp, self.wv = 1.0 / 20, 1.0 / 160

    def initiate(self, z):
        h = z[3]
        mean = np.r_[z, np.zeros(4)]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        h = mean[3]
        q = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(q))

    def update(self, mean, cov, z):
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))
        S = self.H @ cov @ self.H.T + R
        K = np.linalg.solve(S, (cov @ self.H.T).T).T
        mean = mean + K @ (z - self.H @ mean)
        cov = cov - K @ S @ K.T
        return mean, cov


def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / max(h, 1e-6), h], float)


def xyah_to_xyxy(m):
    cx, cy, a, h = m[:4]
    w = a * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], float)


def iou_matrix_np(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def linear_assign(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(cost <= thresh, cost, 1e6)
    r, k = linear_sum_assignment(c)
    m = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mi = {i for i, _ in m}; mj = {j for _, j in m}
    return m, [i for i in range(cost.shape[0]) if i not in mi], [j for j in range(cost.shape[1]) if j not in mj]


class Track:
    _next_id = 0
    TRACKED, LOST, REMOVED = 0, 1, 2

    def __init__(self, box, score, det_idx, kf):
        self.kf, self.box, self.score, self.det_idx = kf, np.asarray(box, float), score, det_idx
        self.mean = self.cov = None
        self.state, self.activated, self.track_id = Track.TRACKED, False, -1
        self.start_frame = self.end_frame = 0

    def predicted_box(self):
        return xyah_to_xyxy(self.mean) if self.mean is not None else self.box

    def predict(self):
        m = self.mean.copy()
        if self.state != Track.TRACKED:
            m[7] = 0.0                          # 놓친 트랙은 높이 변화 속도 0
        self.mean, self.cov = self.kf.predict(m, self.cov)

    def activate(self, frame_id):
        Track._next_id += 1
        self.track_id = Track._next_id
        self.mean, self.cov = self.kf.initiate(xyxy_to_xyah(self.box))
        self.state = Track.TRACKED
        self.activated = frame_id == 1
        self.start_frame = self.end_frame = frame_id

    def update(self, det, frame_id):
        self.mean, self.cov = self.kf.update(self.mean, self.cov, xyxy_to_xyah(det.box))
        self.box, self.score, self.det_idx = det.box, det.score, det.det_idx
        self.state, self.activated, self.end_frame = Track.TRACKED, True, frame_id


class ByteTracker:
    def __init__(self, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,
                 track_buffer=30, match_thresh=0.8, fuse_score=True, frame_rate=30):
        self.hi, self.lo, self.new = track_high_thresh, track_low_thresh, new_track_thresh
        self.max_lost = int(frame_rate / 30.0 * track_buffer)
        self.match, self.fuse = match_thresh, fuse_score
        self.kf = KalmanXYAH()
        self.tracked, self.lost = [], []
        self.frame_id = 0
        Track._next_id = 0

    def _dist(self, tracks, dets, fuse):
        d = 1 - iou_matrix_np([t.predicted_box() for t in tracks], [x.box for x in dets])
        if fuse and len(dets):
            d = 1 - (1 - d) * np.array([x.score for x in dets])[None, :]
        return d

    def update(self, boxes, scores):
        """boxes (N,4) xyxy, scores (N,) → [(track_id, det_idx), ...] 이번 프레임에 확정된 트랙"""
        self.frame_id += 1
        f = self.frame_id
        boxes = np.asarray(boxes, float).reshape(-1, 4); scores = np.asarray(scores, float).reshape(-1)
        high = [Track(boxes[i], scores[i], i, self.kf) for i in np.where(scores >= self.hi)[0]]
        low = [Track(boxes[i], scores[i], i, self.kf) for i in np.where((scores > self.lo) & (scores < self.hi))[0]]

        unconfirmed = [t for t in self.tracked if not t.activated]
        confirmed = [t for t in self.tracked if t.activated]
        pool = confirmed + [t for t in self.lost if t not in confirmed]
        for t in pool:
            t.predict()

        refound, activated_now = [], []
        # 1차: 높은 점수 박스
        m, u_t, u_d = linear_assign(self._dist(pool, high, self.fuse), self.match)
        for i, j in m:
            t = pool[i]
            if t.state != Track.TRACKED:
                refound.append(t)
            t.update(high[j], f); activated_now.append(t)
        # 2차: 남은 '추적 중' 트랙 × 낮은 점수 박스
        r_tracked = [pool[i] for i in u_t if pool[i].state == Track.TRACKED]
        m2, u_t2, _ = linear_assign(self._dist(r_tracked, low, False), 0.5)
        for i, j in m2:
            r_tracked[i].update(low[j], f); activated_now.append(r_tracked[i])
        for i in u_t2:
            r_tracked[i].state = Track.LOST
        # 확정 안 된(1프레임짜리) 트랙 × 남은 높은 점수 박스
        rest_high = [high[j] for j in u_d]
        m3, u_un, u_d3 = linear_assign(self._dist(unconfirmed, rest_high, self.fuse), 0.7)
        for i, j in m3:
            unconfirmed[i].update(rest_high[j], f); activated_now.append(unconfirmed[i])
        for i in u_un:
            unconfirmed[i].state = Track.REMOVED
        # 새 트랙
        new_tracks = []
        for j in u_d3:
            d = rest_high[j]
            if d.score >= self.new:
                d.activate(f); new_tracks.append(d)
        # 오래 못 찾은 트랙 삭제
        for t in pool:
            if t.state == Track.LOST and f - t.end_frame > self.max_lost:
                t.state = Track.REMOVED

        everyone = {id(t): t for t in self.tracked + self.lost + new_tracks}
        self.tracked = [t for t in everyone.values() if t.state == Track.TRACKED]
        self.lost = [t for t in everyone.values() if t.state == Track.LOST]
        return [(t.track_id, t.det_idx) for t in self.tracked if t.activated and t.end_frame == f]


def _bytetrack_selftest():
    """작은 합성 장면으로 동작 확인: 점수가 잠깐 낮아지거나 몇 프레임 사라져도 ID 유지, 약한 새 박스는 새 ID 안 받음"""
    tr = ByteTracker(track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6, track_buffer=30)
    ids = defaultdict(set)
    for f in range(1, 61):
        boxes, scores = [], []
        x = 100 + 3 * f                                     # 선수 A: 오른쪽으로 이동
        if not (30 <= f < 36):                              # 30~35프레임 동안 사라짐(가려짐)
            boxes.append([x, 100, x + 12, 130]); scores.append(0.3 if 15 <= f < 20 else 0.8)   # 15~19 낮은 점수
        boxes.append([300, 200 + f, 312, 230 + f]); scores.append(0.9)                      # 선수 B
        if f == 40:
            boxes.append([500, 50, 512, 80]); scores.append(0.4)                            # 약한 오탐 1번
        for tid, di in tr.update(boxes, scores):
            ids["A" if boxes[di][0] < 290 else ("B" if boxes[di][0] < 400 else "X")].add(tid)
    assert len(ids["A"]) == 1 and len(ids["B"]) == 1 and "X" not in ids, dict(ids)
    print("✅ ByteTrack 자체 점검 통과:", {k: sorted(v) for k, v in ids.items()})

_bytetrack_selftest()

---
# 5b. YOLO26 파인튜닝 + 평가

| 순서 | 내용 | 시간 (T4) |
|---|---|---|
| ① | Roboflow에서 고친 학습 정답(`finetune_gt.zip`) 불러오기 | 1분 |
| ② | 경기장 밖을 회색으로 가린 뒤, 1920 원본을 640 조각으로 잘라 학습 데이터 만들기 | 2분 |
| ③ | **YOLO26 파인튜닝** (사전학습 가중치 → 홍익 / 국민대 / 심판 3클래스) | 25~40분 |
| ④ | 시험지 20장으로 **탐지만** 비교: 사전학습 yolo26x vs 파인튜닝 | 2분 |
| ⑤ | 추적까지 붙여 **v6**으로 채점 → v5와 비교 | 5분 |

- 학습: 튜닝 클립(0143, 0157) 40장만. 채점: 평가 클립(0144, 0156)의 정답 20장만. 둘은 절대 섞지 않아요.
- **경기장 밖을 회색으로 가리는 이유**: 학습 정답에는 경기장 밖 사람(벤치, 코치)에 박스가 없어요. 가리지 않으면 모델이 "사람처럼 생긴 것도 배경"이라고 잘못 배워요.
- **640 조각으로 자르는 이유**: 4단원의 SAHI와 같은 원리로, 선수 크기는 원본 그대로 두고 T4 메모리 안에서 빠르게 학습하려고요. 평가할 때는 1920 전체를 한 번에 넣어요.

**런타임: T4 GPU**

In [ ]:
import time, random, yaml
import torch
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

FT_DIR       = os.path.join(EVAL_DIR, "finetune")
FT_IMG_DIR   = os.path.join(FT_DIR, "images")             # 5a에서 저장한 원본 해상도 프레임
FT_GT_ZIP    = os.path.join(FT_DIR, "finetune_gt.zip")    # Roboflow에서 내보낸 YOLOv8 zip
V5_DIR       = os.path.join(DRIVE_OUTPUT_DIR, "v5")
TEAM = ["Hongik", "Kookmin", "Referee"]
TRAIN_STEMS = ["DJI_20250913133530_0143_D", "DJI_20250913135111_0157_D"]

# 학습 설정
BASE_WEIGHTS = "yolo26m.pt"   # 사전학습 가중치. 시간이 부족하면 "yolo26s.pt" (더 빠름, 덜 정확)
EPOCHS, PATIENCE, BATCH = 60, 15, 16
TILE, STEP_X, STEP_Y = 640, 512, 440   # 1920×1080 → 가로 4 × 세로 2 = 조각 8개 (이웃과 겹침)
SEED = 0
random.seed(SEED); np.random.seed(SEED)

for p in (FT_IMG_DIR, FT_GT_ZIP):
    assert os.path.exists(p), f"없음: {p}\n→ 5a 노트북 실행, Roboflow 내보내기 zip을 이 이름으로 넣었는지 확인하세요."

In [ ]:
# ① 학습 정답 불러오기
FT_GT_DIR = "/content/ft_gt"
shutil.rmtree(FT_GT_DIR, ignore_errors=True)
with zipfile.ZipFile(FT_GT_ZIP) as z:
    z.extractall(FT_GT_DIR)
rf_names = read_names_yaml(FT_GT_DIR)
print("Roboflow 클래스 순서:", rf_names)

FT_GT = {}
for lbl in glob.glob(os.path.join(FT_GT_DIR, "**", "labels", "*.txt"), recursive=True):
    name = original_name(lbl)
    img_path = os.path.join(FT_IMG_DIR, name + ".jpg")
    if not os.path.exists(img_path):
        print("⚠️ 원본 프레임 없음:", name); continue
    H, W = cv2.imread(img_path).shape[:2]
    boxes = []
    for line in open(lbl).read().strip().splitlines():
        if line.strip():
            c, b = yolo_line_to_xyxy(line, W, H)
            if rf_names[c] in TEAM:
                boxes.append((rf_names[c], b))
    FT_GT[name] = {"W": W, "H": H, "boxes": boxes}

assert FT_GT, "학습 정답을 하나도 못 읽었어요. zip 구조를 확인하세요."
assert all(any(n.startswith(s) for s in TRAIN_STEMS) for n in FT_GT), "튜닝 클립이 아닌 이미지가 섞여 있어요"
assert not any(n.startswith(s) for n in FT_GT for s in EVAL_CLIPS), "평가 클립이 학습 정답에 섞여 있어요!"
_c = Counter(c for v in FT_GT.values() for c, _ in v["boxes"])
print(f"학습 정답: 이미지 {len(FT_GT)}장, 박스 {sum(_c.values())}개 {dict(_c)}")

# 5a의 초안과 비교: 얼마나 고쳤나
_pre_n = sum(len(open(os.path.join(FT_DIR, "prelabels", n + ".txt")).read().strip().splitlines())
             for n in FT_GT if os.path.exists(os.path.join(FT_DIR, "prelabels", n + ".txt")))
print(f"초안 박스 {_pre_n}개 → 수정 후 {sum(_c.values())}개 (차이 {sum(_c.values()) - _pre_n:+d})")

In [ ]:
# ② 학습 데이터 만들기: 경기장 밖 가리기 → 640 조각
def v5_poly_at(stem, fid):
    d = json.load(open(os.path.join(V5_DIR, f"{stem}_tracking_final.json"), encoding="utf-8"))
    ch = d.get("field_polygon_by_frame")
    if not ch:
        return np.array(d["field_polygon"], np.int32)
    k = max((int(f) for f in ch if int(f) <= fid), default=None)
    return np.array(ch[str(k)] if k is not None else d["field_polygon"], np.int32)


def mask_outside(img, poly):
    m = np.zeros(img.shape[:2], np.uint8)
    cv2.fillPoly(m, [poly.reshape(-1, 1, 2)], 255)
    out = img.copy(); out[m == 0] = (114, 114, 114)
    return out


def tile_origins(W, H):
    xs = sorted(set(list(range(0, max(1, W - TILE) + 1, STEP_X)) + [max(0, W - TILE)]))
    ys = sorted(set(list(range(0, max(1, H - TILE) + 1, STEP_Y)) + [max(0, H - TILE)]))
    return [(x, y) for y in ys for x in xs]


def boxes_in_tile(boxes, x0, y0, min_keep=0.6):
    """조각 안에 60% 이상 들어온 박스만, 조각 좌표로 잘라서 (경계에 걸린 선수는 이웃 조각에 온전히 들어 있음)"""
    out = []
    for c, (x1, y1, x2, y2) in boxes:
        cx1, cy1, cx2, cy2 = max(x1, x0), max(y1, y0), min(x2, x0 + TILE), min(y2, y0 + TILE)
        if cx2 <= cx1 or cy2 <= cy1:
            continue
        if (cx2 - cx1) * (cy2 - cy1) < min_keep * (x2 - x1) * (y2 - y1) or (cy2 - cy1) < 6:
            continue
        out.append((c, (cx1 - x0, cy1 - y0, cx2 - x0, cy2 - y0)))
    return out


DS = "/content/ft_ds"
shutil.rmtree(DS, ignore_errors=True)
for sp in ("train", "val"):
    os.makedirs(f"{DS}/images/{sp}", exist_ok=True); os.makedirs(f"{DS}/labels/{sp}", exist_ok=True)

names = sorted(FT_GT)
VAL_NAMES = set(names[2::5])               # 5장 중 1장을 검증용 (클립 전체에 고르게)
n_tiles = Counter(); n_boxes = Counter(); dropped_out = 0
for name in names:
    stem, fid = name.split("_f")[0], int(name.split("_f")[-1])
    img = cv2.imread(os.path.join(FT_IMG_DIR, name + ".jpg"))
    poly = v5_poly_at(stem, fid)
    img = mask_outside(img, poly)
    boxes = [(c, b) for c, b in FT_GT[name]["boxes"] if inside(poly, (b[0] + b[2]) / 2, b[3])]
    dropped_out += len(FT_GT[name]["boxes"]) - len(boxes)
    sp = "val" if name in VAL_NAMES else "train"
    H, W = img.shape[:2]
    for (x0, y0) in tile_origins(W, H):
        crop = img[y0:y0 + TILE, x0:x0 + TILE]
        tb = boxes_in_tile(boxes, x0, y0)
        if not tb and (crop == 114).all(axis=2).mean() > 0.9:
            continue                           # 거의 다 가려진 조각은 버림
        tn = f"{name}_x{x0}_y{y0}"
        cv2.imwrite(f"{DS}/images/{sp}/{tn}.jpg", crop)
        open(f"{DS}/labels/{sp}/{tn}.txt", "w").write("\n".join(xyxy_to_yolo_line(TEAM.index(c), b, TILE, TILE) for c, b in tb))
        n_tiles[sp] += 1; n_boxes[sp] += len(tb)

DATA_YAML = f"{DS}/data.yaml"
yaml.safe_dump({"path": DS, "train": "images/train", "val": "images/val", "names": {i: n for i, n in enumerate(TEAM)}},
               open(DATA_YAML, "w"), allow_unicode=True)
print(f"학습 {len(names) - len(VAL_NAMES)}장 → 조각 {n_tiles['train']}개 (박스 {n_boxes['train']}개)")
print(f"검증 {len(VAL_NAMES)}장 → 조각 {n_tiles['val']}개 (박스 {n_boxes['val']}개)")
if dropped_out:
    print(f"참고: 경기장 밖이라 학습에서 뺀 정답 박스 {dropped_out}개")

# 조각 미리보기 2장
_show = sorted(glob.glob(f"{DS}/images/train/*.jpg"))[:: max(1, n_tiles['train'] // 2)][:2]
fig, axes = plt.subplots(1, len(_show), figsize=(12, 6))
for ax, p in zip(np.atleast_1d(axes), _show):
    im = cv2.imread(p)
    for line in open(p.replace("/images/", "/labels/").replace(".jpg", ".txt")).read().strip().splitlines():
        c, b = yolo_line_to_xyxy(line, TILE, TILE)
        cv2.rectangle(im, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), [(255, 140, 0), (40, 40, 230), (0, 215, 255)][c], 2)
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ③ 파인튜닝
# 증강: 좌우 반전·밝기는 켜고, 색조(hue)는 끔 (유니폼 색이 팀 단서라서. 6단원 ResNet18 때와 같은 이유)
model = YOLO(BASE_WEIGHTS)
t0 = time.time()
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, patience=PATIENCE, imgsz=TILE, batch=BATCH, device=DEVICE, seed=SEED,
    hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, fliplr=0.5, degrees=0.0, scale=0.3, mosaic=1.0, close_mosaic=10,
    project="/content/ft_runs", name="yolo26_ft", exist_ok=True, plots=True, verbose=False)
TRAIN_MIN = (time.time() - t0) / 60
RUN_DIR = "/content/ft_runs/yolo26_ft"
BEST = os.path.join(RUN_DIR, "weights", "best.pt")
FT_WEIGHTS = os.path.join(FT_DIR, "yolo26_ft_best.pt")
shutil.copy(BEST, FT_WEIGHTS)                       # 런타임이 끊겨도 남도록 Drive에 저장
shutil.copy(os.path.join(RUN_DIR, "results.csv"), os.path.join(FT_DIR, "train_results.csv"))
print(f"✅ 학습 {TRAIN_MIN:.0f}분 · 가장 좋은 가중치 → {FT_WEIGHTS}")

# 학습 곡선
_r = pd.read_csv(os.path.join(RUN_DIR, "results.csv")); _r.columns = [c.strip() for c in _r.columns]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for col, lab in [("train/box_loss", "학습 박스 손실"), ("val/box_loss", "검증 박스 손실"), ("train/cls_loss", "학습 클래스 손실"), ("val/cls_loss", "검증 클래스 손실")]:
    if col in _r:
        axes[0].plot(_r["epoch"], _r[col], label=lab)
for col, lab in [("metrics/mAP50(B)", "검증 mAP50"), ("metrics/mAP50-95(B)", "검증 mAP50-95"), ("metrics/recall(B)", "검증 Recall")]:
    if col in _r:
        axes[1].plot(_r["epoch"], _r[col], label=lab)
for ax, t in zip(axes, ["손실 (낮을수록 좋음)", "검증 지표 (높을수록 좋음)"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ④ 시험지 20장: 탐지만 비교 (1920 전체 프레임, 1a ROI로 거르기, conf 0.15 — 1b와 같은 조건)
ROI_1A = {}
for stem in EVAL_CLIPS:
    _, ROI_1A[stem] = load_tracking(os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{stem}_tracking_final.json"))
EVAL_NAMES = sorted(GT)
CONF_MAIN = 0.15


def predict_eval(weights, classes=None):
    m = YOLO(weights); out = {}; t0 = time.time()
    for name in EVAL_NAMES:
        r = m.predict(os.path.join(IMG_DIR, name + ".jpg"), imgsz=1920, conf=0.01, classes=classes,
                      max_det=300, device=DEVICE, verbose=False)[0]
        b = r.boxes.xyxy.cpu().numpy().reshape(-1, 4); s = r.boxes.conf.cpu().numpy().reshape(-1)
        c = r.boxes.cls.cpu().numpy().astype(int).reshape(-1)
        poly = ROI_1A[name.split("_f")[0]]
        k = [i for i, bb in enumerate(b) if inside(poly, (bb[0] + bb[2]) / 2, bb[3])]
        out[name] = (b[k], s[k], c[k])
    sec = (time.time() - t0) / len(EVAL_NAMES)
    del m; torch.cuda.empty_cache()
    return out, sec


def det_eval(preds, conf, with_team):
    tp = fp = fn = 0; conf_mat = Counter(); size_tot, size_hit = Counter(), Counter()
    for name in EVAL_NAMES:
        b, s, c = preds[name]; keep = s >= conf
        b, c = b[keep], c[keep]
        g = GT[name]["boxes"]
        pairs, ug, up = match_by_center([bb for _, bb in g], b.tolist())
        tp += len(pairs); fn += len(ug); fp += len(up)
        hit = {gi for gi, _ in pairs}
        for gi, (gc, gb) in enumerate(g):
            sb = "키 20px 미만" if gb[3] - gb[1] < 20 else "20px 이상"
            size_tot[sb] += 1; size_hit[sb] += gi in hit
        if with_team:
            for gi, pi in pairs:
                conf_mat[(g[gi][0], TEAM[c[pi]])] += 1
    P = tp / (tp + fp) if tp + fp else 0; R = tp / (tp + fn) if tp + fn else 0
    row = {"Precision": round(P, 3), "Recall": round(R, 3), "F1": round(2 * P * R / (P + R), 3) if P + R else 0,
           "TP": tp, "FP": fp, "FN": fn, **{f"Recall ({k})": round(size_hit[k] / size_tot[k], 3) for k in size_tot}}
    if with_team:
        tot = sum(v for (gg, _), v in conf_mat.items() if gg in TEAM)
        row["팀 정확도 (박스 단위)"] = round(sum(v for (gg, pp), v in conf_mat.items() if gg == pp) / tot, 3) if tot else None
    return row, conf_mat


P_BASE, sec_b = predict_eval("yolo26x.pt", classes=[0])       # 기준: 사전학습 yolo26x (사람 클래스)
P_FT, sec_f = predict_eval(FT_WEIGHTS)                         # 파인튜닝 (3클래스)
rb, _ = det_eval(P_BASE, CONF_MAIN, False); rf, CM_FT = det_eval(P_FT, CONF_MAIN, True)
DET_CMP = pd.DataFrame({"사전학습 yolo26x (1b와 같음)": {**rb, "초/장": round(sec_b, 2)},
                        f"파인튜닝 {BASE_WEIGHTS.replace('.pt', '')}": {**rf, "초/장": round(sec_f, 2)}})
display(DET_CMP)
print("\n파인튜닝 모델의 팀 분류 혼동행렬 (탐지에서 짝지어진 박스만, 박스 단위):")
display(show_confusion(CM_FT))

# conf 기준별 비교 (파인튜닝 모델은 점수 분포가 달라서 기준을 몇 개 보기)
display(pd.DataFrame({f"conf {c}": det_eval(P_FT, c, True)[0] for c in (0.05, 0.1, 0.15, 0.25, 0.4)}).T[["Precision", "Recall", "F1", "팀 정확도 (박스 단위)"]])

In [ ]:
# ⑤ 추적까지 붙여 v6으로 채점 (v5와 같은 조건: 1a ROI 고정, 2a 추적기 설정, 평가 클립 플레이 구간)
TRACK_PARAMS = json.load(open(os.path.join(EVAL_DIR, "tracker_config_v4.json"), encoding="utf-8"))["params"]
WARMUP = 60                                   # 플레이 시작 60프레임 전부터 추적 시작 (트랙이 자리잡도록)
ft_model = YOLO(FT_WEIGHTS)


def run_clip_ft(stem):
    a, b = EVAL_CLIPS[stem]["play_range"]; start = max(1, a - WARMUP)
    poly = ROI_1A[stem]
    cap = cv2.VideoCapture(video_path_of(stem)); fid = 0
    tracker = ByteTracker(**TRACK_PARAMS, fuse_score=True, frame_rate=30)
    frames, votes = {}, defaultdict(Counter)
    while fid < b:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        if fid < start:
            continue
        r = ft_model.predict(img, imgsz=1920, conf=0.05, max_det=300, device=DEVICE, verbose=False)[0]
        bx = r.boxes.xyxy.cpu().numpy().reshape(-1, 4); sc = r.boxes.conf.cpu().numpy().reshape(-1)
        cl = r.boxes.cls.cpu().numpy().astype(int).reshape(-1)
        k = [i for i, bb in enumerate(bx) if inside(poly, (bb[0] + bb[2]) / 2, bb[3])]
        bx, sc, cl = bx[k], sc[k], cl[k]
        rows = []
        for tid, di in tracker.update(bx, sc):
            votes[int(tid)][int(cl[di])] += float(sc[di])     # 트랙의 팀 = 점수 가중 다수결
            rows.append((int(tid), bx[di], float(sc[di])))
        frames[fid] = rows
    cap.release()
    lab = {t: TEAM[v.most_common(1)[0][0]] for t, v in votes.items()}
    out = []
    for f in sorted(frames):
        ps = []
        for tid, bb, s in frames[f]:
            x1, y1, x2, y2 = [int(round(v)) for v in bb]
            ps.append({"track_id": tid, "type": lab.get(tid, "Unknown"), "bbox": [x1, y1, x2, y2], "feet": [(x1 + x2) // 2, y2], "conf": s})
        out.append({"frame_id": f, "players": ps})
    d = {"video_name": os.path.basename(video_path_of(stem)), "fps": 30, "field_polygon": np.asarray(poly).tolist(), "frames": out,
         "notes": [f"v6: 파인튜닝 {BASE_WEIGHTS} 3클래스 탐지 + 2a ByteTrack + 트랙 다수결 팀 (ResNet18 없음)"]}
    p = os.path.join(EVAL_DIR, "runs", "v6_yolo_ft", f"{stem}_tracking_final.json")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    json.dump(d, open(p, "w", encoding="utf-8"), ensure_ascii=False)
    return p


t0 = time.time()
V6 = {stem: run_clip_ft(stem) for stem in EVAL_CLIPS}
print(f"v6 처리 {time.time() - t0:.0f}s")
_, CONF_V6, _ = evaluate_run("v6_yolo_ft", V6, note=f"파인튜닝 {BASE_WEIGHTS} (튜닝 클립 40장, 3클래스) + 2a 추적 + 트랙 다수결")

res = pd.read_csv(RESULTS_CSV).set_index("run")
order = [r for r in ["v3_baseline", "v5_cnn", "v6_yolo_ft"] if r in res.index]
display(res.loc[order, ["det_precision", "det_recall", "det_f1", "TP", "FP", "FN", "team_acc", "referee_recall"]])
print("\n=== v6 혼동행렬 ===")
display(show_confusion(CONF_V6))

## 결과 공유
1. ①의 학습 정답 개수 (초안 → 수정 후)와 ②의 조각 개수
2. ③의 학습 시간과 학습 곡선
3. ④의 비교표 2개와 혼동행렬
4. ⑤의 비교표 (v3 / v5 / v6)와 v6 혼동행렬

**읽는 법**
- ④ Recall이 사전학습보다 오르면: 직접 추가한 ‘엉킨 선수’를 모델이 배운 것
- ④ 팀 정확도: YOLO 하나로 팀까지 맞힐 수 있는지 (v5는 ResNet18이 따로 맡음)
- ⑤ v5와 비교: 파이프라인 전체에서 나아졌는지. 학습 데이터가 40장뿐이라 나빠질 수도 있고, 그것도 의미 있는 결과예요.